# KDE bandwidth playground and ISE benchmark

This notebook contains introductory KDE plots and an oracle integrated-squared-error (ISE) benchmark for simulated Gaussian samples.

1. Direct KDE visualization for chosen sample and bandwidth parameters.
2. Comparisons of several bandwidths on the same sample.
3. An interactive KDE plot.
4. An oracle bandwidth computed from the known density in simulation.

The pairwise beta-2/$L^2$ divergence experiments are maintained in
[`spectral_scale_selection_kde.ipynb`](spectral_scale_selection_kde.ipynb).

Sources for the Gaussian KDE and exact ISE benchmark:
[shared density helpers](kde_tools/density.py),
[Gaussian density helper](kde_tools/gaussian_mixtures.py).


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib-cache"))

import numpy as np
import matplotlib.pyplot as plt                                                                     ### PLOTTING

from kde_tools.pdf_export import enable_automatic_pdf_saving                                        ### PLOTTING
from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta
from kde_tools.density import exact_oracle_ise_values, gaussian_kde_values
from kde_tools.gaussian_mixtures import gaussian_density_with_variance
from kde_tools.quadrature import quadrature_grid_for_kdes

pdf_output_directory = enable_automatic_pdf_saving(                                                 ### PLOTTING
    "100_kde_test"                                                                                  ### PLOTTING
)                                                                                                   ### PLOTTING

plt.style.use("seaborn-v0_8-whitegrid")


## 1. Kernel functions and KDE implementation

The first code block defines the two kernels used throughout the notebook: Gaussian and Epanechnikov. Each kernel $K$ is normalized so that

$$

\int_{\mathbb R} K(u)\,du=1.

$$

For bandwidth $h$, the rescaled kernel is

$$

K_h(x)=\frac{1}{h}K\left(\frac{x}{h}\right),

$$

and therefore the KDE can also be written as

$$

\widehat{f}_h(x)=\frac{1}{n}\sum_{r=1}^n K_h(x-X_r).

$$

In the code, `kde_eval(x_grid, samples, h, kernel)` supports both kernels for the introductory plots. The oracle benchmark below uses the shared Gaussian-KDE helpers from `kde_tools`.


In [ ]:
AVAILABLE_KERNELS = [
    "gaussian",
    "epanechnikov",
]


def kernel_values(u, kernel="gaussian"):
    """Evaluate a normalized one-dimensional Gaussian or Epanechnikov kernel K(u)."""
    u = np.asarray(u)
    kernel = kernel.lower()

    if kernel == "gaussian":
        return np.exp(-0.5 * u**2) / np.sqrt(2 * np.pi)

    if kernel == "epanechnikov":
        out = np.zeros_like(u, dtype=float)
        inside = np.abs(u) <= 1
        out[inside] = 0.75 * (1 - u[inside] ** 2)
        return out

    raise ValueError(f"Unknown kernel: {kernel!r}. Choose one of {AVAILABLE_KERNELS}.")


def kde_eval(x_grid, samples, h, kernel="gaussian"):
    """Evaluate the KDE on x_grid for data samples and bandwidth h."""
    samples = np.asarray(samples)
    x_grid = np.asarray(x_grid)

    if h <= 0:
        raise ValueError("The bandwidth h must be positive.")
    if len(samples) == 0:
        raise ValueError("Need at least one sample point.")

    u = (x_grid[:, None] - samples[None, :]) / h
    return kernel_values(u, kernel=kernel).mean(axis=1) / h


## 2. Static KDE example

This block samples data once and computes one KDE for one manually chosen bandwidth. It is the simplest sanity check for the estimator.

The parameters are:

- `n`: number of sample points;
- `sigma2`: variance of the true normal distribution;
- `h`: KDE bandwidth;
- `kernel`: kernel shape;
- `seed`: random seed, so the experiment is reproducible.

The plot overlays the histogram, the KDE $\widehat{f}_h$, the true density $f$, and the sample locations. This lets you visually inspect whether the chosen bandwidth undersmooths or oversmooths.


In [ ]:
# Parameters you can change
n = 200
sigma2 = 1.0
h = 0.25
kernel = "gaussian"
seed = 7

rng = np.random.default_rng(seed)
X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n)

x_min = X.min() - 4 * h - 0.5
x_max = X.max() + 4 * h + 0.5
x_grid = np.linspace(x_min, x_max, 800)

kde = kde_eval(x_grid, X, h=h, kernel=kernel)
truth = gaussian_density_with_variance(x_grid, mean=0.0, variance=sigma2)

fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
ax.hist(X, bins="auto", density=True, alpha=0.25, color="gray", label="sample histogram")           ### PLOTTING
ax.plot(x_grid, kde, lw=2.5, label=f"KDE, kernel={kernel}, h={h}")                                  ### PLOTTING
ax.plot(x_grid, truth, "--", lw=2, label=fr"true $N(0,{sigma2})$ density")                          ### PLOTTING
ax.scatter(X, np.zeros_like(X), marker="|", s=80, color="black", alpha=0.45, label="sample points") ### PLOTTING
ax.set_title("Kernel density estimate")
ax.set_xlabel("x")                                                                                  ### PLOTTING
ax.set_ylabel("density")                                                                            ### PLOTTING
ax.legend()                                                                                         ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 3. Compare several bandwidths

Here we hold the data set fixed and evaluate several KDEs

$$

\widehat{f}_{h_1},\widehat{f}_{h_2},\dots

$$

on the same grid. This isolates the effect of the bandwidth from the randomness of the sample.

The qualitative behavior should be:

- small $h$: many narrow peaks, high variance, low bias;
- large $h$: one broad smooth curve, low variance, high bias;
- intermediate $h$: often the best compromise.

This is the visual intuition behind bandwidth selection.


In [ ]:
n = 200
sigma2 = 1.0
kernel = "gaussian"
bandwidths = [0.08, 0.15, 0.30, 0.60, 1.20]
seed = 7

rng = np.random.default_rng(seed)
X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n)
x_grid = np.linspace(X.min() - 2.5, X.max() + 2.5, 1000)

fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
ax.hist(X, bins="auto", density=True, alpha=0.18, color="gray", label="sample histogram")           ### PLOTTING
ax.plot(x_grid, gaussian_density_with_variance(x_grid, mean=0.0, variance=sigma2), "k--", lw=2, label="true density")  ### PLOTTING

for h_value in bandwidths:
    kde = kde_eval(x_grid, X, h=h_value, kernel=kernel)
    ax.plot(x_grid, kde, lw=2, label=f"h={h_value:g}")                                              ### PLOTTING

ax.set_title(f"Bandwidth comparison using the {kernel} kernel")
ax.set_xlabel("x")                                                                                  ### PLOTTING
ax.set_ylabel("density")                                                                            ### PLOTTING
ax.legend(ncol=2)                                                                                   ### PLOTTING
plt.show()                                                                                          ### PLOTTING


## 4. Interactive KDE plot

This section repeats the static KDE experiment with sliders. The most important control is the bandwidth slider $h$. Moving it shows the scale-space behavior of the KDE in real time.

Mathematically, changing $h$ changes the amount of smoothing in

$$

\widehat{f}_h(x)=\frac{1}{nh}\sum_{r=1}^nK\left(\frac{x-X_r}{h}\right).

$$

The sample itself changes only when `seed`, `n`, or `sigma²` changes. If `ipywidgets` is unavailable, the rest of the notebook still works; only this interactive control block is skipped.


In [ ]:
try:
    from ipywidgets import Button, Dropdown, FloatLogSlider, HBox, IntSlider, Output, VBox
    from IPython.display import display                                                             ### REPORTING
    widgets_available = True
except ModuleNotFoundError:
    widgets_available = False
    print(                                                                                          ### REPORTING
        "ipywidgets is not installed in this Python environment. "                                  ### REPORTING
        "Install it with `pip install ipywidgets` or `conda install ipywidgets` "                   ### REPORTING
        "to use the interactive sliders. The non-interactive KDE and oracle cells still work."      ### REPORTING
    )                                                                                               ### REPORTING


if widgets_available:
    n_slider = IntSlider(value=200, min=20, max=2000, step=20, description="n")
    sigma2_slider = FloatLogSlider(
        value=1.0,
        base=10,
        min=-2,
        max=1.3,
        step=0.05,
        description="sigma²",
        readout_format=".3g",
    )
    seed_slider = IntSlider(value=7, min=0, max=999, step=1, description="seed")
    kernel_dropdown = Dropdown(options=AVAILABLE_KERNELS, value="gaussian", description="kernel")
    h_slider = FloatLogSlider(
        value=0.25,
        base=10,
        min=-2,
        max=1.1,
        step=0.01,
        description="h",
        readout_format=".3g",
        continuous_update=True,
    )
    resample_button = Button(description="resample", button_style="")
    out = Output()

    def draw_interactive_kde(*_):
        n = n_slider.value
        sigma2 = sigma2_slider.value
        seed = seed_slider.value
        h = h_slider.value
        kernel = kernel_dropdown.value

        rng = np.random.default_rng(seed)
        X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n)

        # Use a plotting window that adapts to both the data spread and the current bandwidth.
        margin = max(4 * np.sqrt(sigma2), 6 * h, 1.0)
        x_grid = np.linspace(X.min() - margin, X.max() + margin, 1000)
        kde = kde_eval(x_grid, X, h=h, kernel=kernel)
        truth = gaussian_density_with_variance(x_grid, mean=0.0, variance=sigma2)

        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 5.5))                                               ### PLOTTING
            ax.hist(X, bins="auto", density=True, alpha=0.20, color="gray", label="sample histogram")  ### PLOTTING
            ax.plot(x_grid, kde, lw=2.5, color="tab:blue", label=f"KDE, h={h:.3g}")                 ### PLOTTING
            ax.plot(x_grid, truth, "--", lw=2, color="black", label=fr"true $N(0,{sigma2:.3g})$ density")  ### PLOTTING
            ax.scatter(X, np.zeros_like(X), marker="|", s=65, color="black", alpha=0.35)            ### PLOTTING
            ax.set_title(f"Interactive KDE — {kernel} kernel, n={n}, sigma²={sigma2:.3g}")
            ax.set_xlabel("x")                                                                      ### PLOTTING
            ax.set_ylabel("density")                                                                ### PLOTTING
            ax.legend(loc="upper right")                                                            ### PLOTTING
            plt.show()                                                                              ### PLOTTING

    def advance_seed(_):
        seed_slider.value = seed_slider.value + 1 if seed_slider.value < seed_slider.max else seed_slider.min

    for widget in [n_slider, sigma2_slider, seed_slider, kernel_dropdown, h_slider]:
        widget.observe(draw_interactive_kde, names="value")

    resample_button.on_click(advance_seed)

    top_controls = HBox([n_slider, sigma2_slider])
    middle_controls = HBox([kernel_dropdown, seed_slider, resample_button])

    display(VBox([top_controls, middle_controls, out, h_slider]))                                   ### REPORTING
    draw_interactive_kde()


## 5. Shared simulation setup and numerical helper formulas

From here on the notebook uses one common simulated sample

$$
X_1,\dots,X_n\sim N(0,\sigma^2),
$$

a bandwidth grid

$$
h_1<\cdots<h_N,
$$

and the corresponding KDE family

$$
\widehat f_{h_1},\dots,\widehat f_{h_N}.
$$

The densities and integrated squared errors are evaluated on a common numerical integration grid. Integrated squared error is retained only as an oracle evaluation benchmark; it is not used here to build a pairwise spectral similarity.


In [ ]:
# Shared state used by the ISE benchmark below.
n_scale = 200
sigma2 = 1.0
seed_scale = 123
n_h = 80

rng = np.random.default_rng(seed_scale)
X = rng.normal(loc=0.0, scale=np.sqrt(sigma2), size=n_scale)

# The grid is centered around the usual n^{-1/5} bandwidth scale, but spans
# substantially smaller and larger bandwidths.
h_center = np.sqrt(sigma2) * n_scale ** (-1 / 5)
h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=0.20 * h_center,
    h_max=3.00 * h_center,
    beta=2.0,
    grid_size=n_h,
)

x_grid = quadrature_grid_for_kdes(
    samples=X,
    h_grid=h_grid,
    m_x=2_001,
)
kde_family = gaussian_kde_values(
    samples=X,
    h_grid=h_grid,
    x_grid=x_grid,
)
truth = gaussian_density_with_variance(
    x=x_grid,
    mean=0.0,
    variance=sigma2,
)

# Exact Gaussian-KDE ISE benchmark against the known N(0, sigma^2) density.
ise_values = exact_oracle_ise_values(
    samples=X,
    h_grid=h_grid,
    sigma2=sigma2,
)
oracle_index = int(np.argmin(ise_values))
oracle_h = float(h_grid[oracle_index])

print("Shared KDE state rebuilt")                                                                   ### REPORTING
print(f"sample size n = {n_scale}")                                                                 ### REPORTING
print("kernel = gaussian")                                                                          ### REPORTING
print(f"oracle h = {oracle_h:.6g}")                                                                 ### REPORTING


## 6. Oracle bandwidth from the known true density

Because the data in this notebook are simulated from a known density

$$
f(x)=N(0,\sigma^2),
$$

we can compute an oracle benchmark. For every candidate bandwidth, we evaluate

$$
ISE(h)=\int(\widehat f_h(x)-f(x))^2\,dx.
$$

The oracle bandwidth is

$$
h_{oracle}=\arg\min_{h_i} ISE(h_i).
$$

This is not available for real data; it is only a benchmark for the other selectors.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))                                                             ### PLOTTING
ax.plot(h_grid, ise_values, marker="o", ms=4, lw=2, label="ISE curve")                              ### PLOTTING
ax.axvline(oracle_h, color="tab:green", ls="-", lw=2, label=f"oracle h={oracle_h:.3g}")             ### PLOTTING
ax.set_xscale("log")                                                                                ### PLOTTING
ax.set_title("Oracle bandwidth via integrated squared error")
ax.set_xlabel("bandwidth h")                                                                        ### PLOTTING
ax.set_ylabel("ISE(h)")                                                                             ### PLOTTING
ax.legend()                                                                                         ### PLOTTING
plt.show()                                                                                          ### PLOTTING
